# Notebook 3 — Algorithmes Machine Learning
### BNP Paribas — Senior AI Engineer

---
## Sommaire
1. [Régression linéaire & régularisation](#regression)
2. [Classification (Logistic, SVM, KNN, Naive Bayes)](#classification)
3. [Arbres de décision](#trees)
4. [Clustering (K-Means, DBSCAN, GMM)](#clustering)
5. [Bagging & Random Forest](#bagging)
6. [Boosting (AdaBoost, GBM, XGBoost)](#boosting)
7. [Métriques & évaluation](#metrics)
8. [Questions d’entretien](#questions)


---
## 1. Régression <a name='regression'></a>

### Régression linéaire
- **Equation normale** : `theta = (X^T X)^-1 X^T y`  — solution exacte O(n^3)
- **Gradient descent** : itératif, scalable aux gros datasets
- **Hypothèses OLS** (Gauss-Markov) : linearité, homoscedasticite, independance, normalité des residus

### Régularisation
| Methode | Penalite | Effet |
|---|---|---|
| Ridge (L2) | `lambda * sum(w^2)` | Shrink les coefficients, reduit la variance |
| Lasso (L1) | `lambda * sum(|w|)` | Sparse : annule certains coefficients (selection de features) |
| ElasticNet | `alpha*L1 + (1-alpha)*L2` | Combine les deux, utile si features correlees |

> **Lien MAP** : Ridge = MAP avec prior gaussien, Lasso = MAP avec prior laplacien


In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import warnings; warnings.filterwarnings("ignore")

np.random.seed(42)
n = 200
X_raw = np.sort(np.random.uniform(0, 10, n))
y = 2.5 * X_raw + np.sin(X_raw) * 3 + np.random.randn(n) * 1.5

X = X_raw.reshape(-1, 1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ============================================================
# EQUATION NORMALE -- from scratch
# ============================================================
print("=== Equation Normale (OLS) from scratch ===")
X_b = np.column_stack([X_train.ravel(), np.ones(len(X_train))])
# theta = (X^T X)^-1 X^T y
theta = np.linalg.lstsq(X_b, y_train, rcond=None)[0]
y_pred_scratch = np.column_stack([X_test.ravel(), np.ones(len(X_test))]) @ theta
print(f"  w={theta[0]:.4f}, b={theta[1]:.4f}")
print(f"  R2={r2_score(y_test, y_pred_scratch):.4f}")
print(f"  RMSE={np.sqrt(mean_squared_error(y_test, y_pred_scratch)):.4f}")
print(f"  MAE={mean_absolute_error(y_test, y_pred_scratch):.4f}")

# ============================================================
# REGULARISATION -- comparaison
# ============================================================
print("\n=== Ridge vs Lasso vs ElasticNet ===")
# Dataset avec 20 features dont 5 vraiment utiles
np.random.seed(42)
X_reg = np.random.randn(200, 20)
y_reg = (X_reg[:, 0]*3 + X_reg[:, 2]*(-1.5) + X_reg[:, 5]*2
         + X_reg[:, 8]*0.5 + X_reg[:, 12]*(-1) + np.random.randn(200)*0.5)

X_tr, X_te, y_tr, y_te = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s = scaler.transform(X_te)

models = [
    ("OLS (MLE)",          LinearRegression()),
    ("Ridge (L2, a=1)",    Ridge(alpha=1.0)),
    ("Ridge (L2, a=10)",   Ridge(alpha=10.0)),
    ("Lasso (L1, a=0.1)",  Lasso(alpha=0.1)),
    ("Lasso (L1, a=0.5)",  Lasso(alpha=0.5)),
    ("ElasticNet",         ElasticNet(alpha=0.1, l1_ratio=0.5)),
]
for name, model in models:
    model.fit(X_tr_s, y_tr)
    r2  = r2_score(y_te, model.predict(X_te_s))
    nz  = (np.abs(model.coef_) > 0.01).sum()
    l2  = np.linalg.norm(model.coef_)
    print(f"  {name:25s}: R2={r2:.4f}  coefs_non0={nz:2d}/20  ||w||2={l2:.2f}")


In [ ]:
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.model_selection import validation_curve
import warnings; warnings.filterwarnings("ignore")

np.random.seed(42)
X = np.random.randn(200, 10)
y = X[:,0]*3 + X[:,2]*(-1.5) + np.random.randn(200)*0.5

# ============================================================
# LEARNING CURVE -- diagnostiquer biais/variance
# ============================================================
from sklearn.model_selection import learning_curve
from sklearn.linear_model import LinearRegression

print("=== Learning Curve (diagnostiquer biais / variance) ===")
train_sizes, train_scores, val_scores = learning_curve(
    LinearRegression(), X, y, cv=5, scoring="r2",
    train_sizes=np.linspace(0.1, 1.0, 8)
)
print(f"  {'n_train':>8}  {'train_R2':>10}  {'val_R2':>10}")
for size, tr, val in zip(train_sizes, train_scores.mean(1), val_scores.mean(1)):
    print(f"  {size:8d}  {tr:10.4f}  {val:10.4f}")

# ============================================================
# VALIDATION CURVE -- tuner alpha de Ridge
# ============================================================
print("\n=== Validation Curve (Ridge alpha) ===")
alphas = [0.001, 0.01, 0.1, 1, 10, 100]
train_s, val_s = validation_curve(
    Ridge(), X, y, param_name="alpha", param_range=alphas, cv=5, scoring="r2"
)
print(f"  {'alpha':>8}  {'train_R2':>10}  {'val_R2':>10}")
for a, tr, val in zip(alphas, train_s.mean(1), val_s.mean(1)):
    print(f"  {a:8.3f}  {tr:10.4f}  {val:10.4f}")
print("  => Choisir l alpha avec le meilleur val_R2 (compromis biais/variance)")


---
## 2. Classification <a name='classification'></a>

### Regression Logistique
- `P(y=1|x) = sigmoid(w^T x + b) = 1 / (1 + exp(-z))`
- Loss = **cross-entropie** : `-[y log(p) + (1-y) log(1-p)]`
- Optimisé par gradient descent (pas de solution analytique)
- Multi-classe : OvR (One-vs-Rest) ou Softmax

### SVM (Support Vector Machine)
- Maximise la **marge** entre les classes : `2 / ||w||`
- Support vectors = points les plus proches de l’hyperplan
- **Kernel trick** : projetation implicite en haute dimension
  - RBF : `K(x,z) = exp(-gamma * ||x-z||^2)` (le plus utilisé)
  - Polynomial : `K(x,z) = (x^T z + c)^d`
- Param : **C** (compromis marge/erreurs), **gamma** (largeur RBF)

### Naive Bayes
- `P(y|x) proportionnel a P(y) * prod P(xi|y)` — hypothese d'independance
- Tres rapide, bon sur le texte, peu de données


In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score
import warnings; warnings.filterwarnings("ignore")

np.random.seed(42)
X, y = make_classification(n_samples=1000, n_features=10, n_informative=6,
                            n_redundant=2, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
sc = StandardScaler()
X_tr_s, X_te_s = sc.fit_transform(X_tr), sc.transform(X_te)

# ============================================================
# LOGISTIC REGRESSION -- from scratch (sigmoid + gradient)
# ============================================================
def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def log_loss(y_true, y_pred):
    y_pred = np.clip(y_pred, 1e-10, 1 - 1e-10)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

def logistic_regression_gd(X, y, lr=0.1, epochs=200):
    n, d = X.shape
    w = np.zeros(d); b = 0.0
    for _ in range(epochs):
        z = X @ w + b
        p = sigmoid(z)
        dw = (X.T @ (p - y)) / n
        db = np.mean(p - y)
        w -= lr * dw
        b -= lr * db
    return w, b

print("=== Logistic Regression from scratch ===")
w, b = logistic_regression_gd(X_tr_s, y_tr)
p_pred = sigmoid(X_te_s @ w + b)
y_pred_scratch = (p_pred >= 0.5).astype(int)
print(f"  Accuracy : {accuracy_score(y_te, y_pred_scratch):.4f}")
print(f"  AUC-ROC  : {roc_auc_score(y_te, p_pred):.4f}")

# ============================================================
# COMPARAISON DES CLASSIFIEURS
# ============================================================
print("\n=== Comparaison classifieurs (CV=5) ===")
models = {
    "LogisticRegression":  LogisticRegression(max_iter=1000),
    "SVM (linear)":        SVC(kernel="linear", probability=True),
    "SVM (RBF)":           SVC(kernel="rbf", C=1.0, probability=True),
    "KNN (k=5)":           KNeighborsClassifier(n_neighbors=5),
    "Naive Bayes":         GaussianNB(),
}
for name, model in models.items():
    scores = cross_val_score(model, X_tr_s, y_tr, cv=5, scoring="roc_auc")
    print(f"  {name:25s}: AUC={scores.mean():.4f} (+/- {scores.std():.4f})")


In [ ]:
import numpy as np
from sklearn.metrics import (confusion_matrix, classification_report,
                              roc_curve, auc, precision_recall_curve,
                              average_precision_score)
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import warnings; warnings.filterwarnings("ignore")

np.random.seed(42)
# Dataset desequilibre (typique en finance/fraude)
X, y = make_classification(n_samples=2000, n_features=10, n_informative=5,
                            weights=[0.90, 0.10], random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
sc = StandardScaler()
X_tr_s, X_te_s = sc.fit_transform(X_tr), sc.transform(X_te)

model = LogisticRegression(max_iter=1000, class_weight="balanced").fit(X_tr_s, y_tr)
y_pred = model.predict(X_te_s)
y_proba = model.predict_proba(X_te_s)[:, 1]

# ============================================================
# METRIQUES DETAILLEES
# ============================================================
cm = confusion_matrix(y_te, y_pred)
TP, FP, FN, TN = cm[1,1], cm[0,1], cm[1,0], cm[0,0]

print("=== Matrice de confusion ===")
print(f"  TN={TN}  FP={FP}")
print(f"  FN={FN}  TP={TP}")

precision = TP / (TP + FP + 1e-9)
recall    = TP / (TP + FN + 1e-9)
f1        = 2 * precision * recall / (precision + recall + 1e-9)
accuracy  = (TP + TN) / cm.sum()

print(f"\n  Accuracy  = {accuracy:.4f}  (attention: trompeuse sur classes desequilibrees!)")
print(f"  Precision = {precision:.4f}  (parmi les positifs predits, combien sont vrais ?)")
print(f"  Recall    = {recall:.4f}  (parmi les vrais positifs, combien retrouves ?)")
print(f"  F1        = {f1:.4f}  (harmonic mean precision/recall)")

fpr, tpr, _ = roc_curve(y_te, y_proba)
auc_roc = auc(fpr, tpr)
pr_auc = average_precision_score(y_te, y_proba)
print(f"  AUC-ROC   = {auc_roc:.4f}  (robuste au desequilibre de classes)")
print(f"  PR-AUC    = {pr_auc:.4f}  (encore plus adapte aux classes rares)")

print("\n=== Quand utiliser quelle metrique ? ===")
print("  Detection de fraude      -> Recall (ne pas manquer de fraude)")
print("  Spam filter              -> Precision (ne pas bloquer de vrais emails)")
print("  Classes desequilibrees   -> F1, PR-AUC")
print("  Comparaison generale     -> AUC-ROC (independant du seuil)")
print("  Reporting metier         -> Accuracy SEULEMENT si classes equilibrees")


---
## 3. Arbres de Décision <a name='trees'></a>

### Principe de construction (CART)
1. Pour chaque feature et chaque seuil possible, calculer la **reduction d'impurete**
2. Choisir le split qui maximise le **gain d'information**
3. Répéter récursivement jusqu’à un critère d’arrêt

### Criteres d'impurete
- **Gini** : `1 - sum(p_i^2)` — plus rapide
- **Entropie** : `-sum(p_i log p_i)` — plus informé
- **MSE** (régression) : variance intra-noeud

### Hyperparaetres et overfitting
| Param | Effet | Conseil |
|---|---|---|
| `max_depth` | Profondeur max | 3-5 comme point de depart |
| `min_samples_split` | Min samples pour splitter | Augmenter pour regulariser |
| `min_samples_leaf` | Min samples par feuille | Au moins 5-10 |
| `max_features` | Features candidates par split | sqrt(p) pour RF |
| `ccp_alpha` | Elagage post-hoc (pruning) | Tuner par CV |


In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, export_text
from sklearn.datasets import load_iris, make_regression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score
import warnings; warnings.filterwarnings("ignore")

# ============================================================
# SPLIT CRITERION -- from scratch
# ============================================================
def gini(y):
    if len(y) == 0: return 0.0
    _, counts = np.unique(y, return_counts=True)
    p = counts / len(y)
    return 1 - np.sum(p**2)

def entropy(y):
    if len(y) == 0: return 0.0
    _, counts = np.unique(y, return_counts=True)
    p = counts / len(y)
    p = p[p > 0]
    return -np.sum(p * np.log2(p + 1e-15))

def information_gain(y_parent, y_left, y_right, criterion="gini"):
    metric = gini if criterion == "gini" else entropy
    n = len(y_parent)
    weighted_child = (len(y_left)/n * metric(y_left) +
                      len(y_right)/n * metric(y_right))
    return metric(y_parent) - weighted_child

def best_split(X, y, criterion="gini"):
    best_gain, best_feat, best_thresh = -1, None, None
    for feat in range(X.shape[1]):
        thresholds = np.unique(X[:, feat])
        for thresh in thresholds:
            left_mask = X[:, feat] <= thresh
            if left_mask.sum() == 0 or (~left_mask).sum() == 0:
                continue
            gain = information_gain(y, y[left_mask], y[~left_mask], criterion)
            if gain > best_gain:
                best_gain, best_feat, best_thresh = gain, feat, thresh
    return best_feat, best_thresh, best_gain

iris = load_iris()
X, y = iris.data, iris.target
feat, thresh, gain = best_split(X, y, "gini")
print("=== Meilleur split (Gini) sur Iris ===")
print(f"  Feature={iris.feature_names[feat]}, threshold={thresh:.2f}, gain={gain:.4f}")

feat_e, thresh_e, gain_e = best_split(X, y, "entropy")
print(f"  Feature={iris.feature_names[feat_e]}, threshold={thresh_e:.2f}, gain_entropy={gain_e:.4f}")

# ============================================================
# BIAIS / VARIANCE selon max_depth
# ============================================================
print("\n=== Biais / Variance selon max_depth ===")
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"  {'max_depth':>10}  {'train_acc':>10}  {'test_acc':>10}  {'n_leaves':>8}")
for d in [1, 2, 3, 5, 8, None]:
    dt = DecisionTreeClassifier(max_depth=d, random_state=42)
    dt.fit(X_tr, y_tr)
    tr_acc = accuracy_score(y_tr, dt.predict(X_tr))
    te_acc = accuracy_score(y_te, dt.predict(X_te))
    print(f"  {str(d):>10}  {tr_acc:>10.4f}  {te_acc:>10.4f}  {dt.get_n_leaves():>8d}")

print("  => depth=None -> overfitting (train=1.0, test degrade)")


In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
import warnings; warnings.filterwarnings("ignore")

iris = load_iris()
X, y = iris.data, iris.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

# ============================================================
# PRUNING -- Cost Complexity Pruning (post-hoc)
# ============================================================
print("=== Cost Complexity Pruning (ccp_alpha) ===")
dt_full = DecisionTreeClassifier(random_state=42).fit(X_tr, y_tr)
path = dt_full.cost_complexity_pruning_path(X_tr, y_tr)
ccp_alphas = path.ccp_alphas

from sklearn.model_selection import cross_val_score
print(f"  {'ccp_alpha':>12}  {'train_acc':>10}  {'cv_acc':>10}  {'n_leaves':>8}")
for alpha in ccp_alphas[::max(1, len(ccp_alphas)//8)]:
    dt = DecisionTreeClassifier(ccp_alpha=alpha, random_state=42).fit(X_tr, y_tr)
    tr = dt.score(X_tr, y_tr)
    cv = cross_val_score(dt, X_tr, y_tr, cv=5).mean()
    print(f"  {alpha:>12.4f}  {tr:>10.4f}  {cv:>10.4f}  {dt.get_n_leaves():>8d}")

# Visualisation texte de l arbre
print("\n=== Structure de l arbre (depth=3) ===")
dt_viz = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_tr, y_tr)
print(export_text(dt_viz, feature_names=iris.feature_names))


---
## 4. Clustering <a name='clustering'></a>

### K-Means
- Minimise l'inertie : `sum_k sum_{x in C_k} ||x - mu_k||^2`
- Algorithme EM : Expectation (assign) / Maximisation (update centroides)
- **K-Means++** : initialisation intelligente pour eviter les minima locaux
- Hypotheses : clusters **spheriques**, de **taille similaire**

### DBSCAN
- Clustering base sur la **densite**
- Parametres : **eps** (rayon de voisinage), **min_samples** (min points pour un core point)
- Detecte automatiquement les **outliers** (label = -1)
- Pas besoin de specifier k, clusters de forme arbitraire

### GMM (Gaussian Mixture Model)
- Modele probabiliste : `P(x) = sum_k pi_k * N(x; mu_k, Sigma_k)`
- **Soft assignment** : probabilite d'appartenance a chaque cluster
- Algorithme **EM** pour estimer les parametres
- **BIC/AIC** pour choisir le nombre de composantes

### Metriques
| Metrique | Formule | Ideal |
|---|---|---|
| Inertie | sum dist(x, centroide)^2 | Minimiser |
| Silhouette | (b-a)/max(a,b) | Max (proche de 1) |
| Davies-Bouldin | ratio cohesion/separation | Minimiser |


In [ ]:
import numpy as np
from sklearn.cluster import KMeans, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.datasets import make_blobs, make_moons
from sklearn.preprocessing import StandardScaler
import warnings; warnings.filterwarnings("ignore")

np.random.seed(42)

# ============================================================
# K-MEANS -- from scratch avec K-Means++ init
# ============================================================
def kmeans_plus_plus_init(X, k, rng):
    n = len(X)
    centroids = [X[rng.randint(n)]]
    for _ in range(k - 1):
        dists = np.array([min(np.linalg.norm(x - c)**2 for c in centroids) for x in X])
        probs = dists / dists.sum()
        centroids.append(X[rng.choice(n, p=probs)])
    return np.array(centroids)

def kmeans(X, k, max_iter=300, seed=42):
    rng = np.random.RandomState(seed)
    centroids = kmeans_plus_plus_init(X, k, rng)
    labels = np.zeros(len(X), dtype=int)

    for it in range(max_iter):
        # E-step : assigner chaque point au centroide le plus proche
        dists = np.linalg.norm(X[:, None] - centroids[None, :], axis=2)
        new_labels = np.argmin(dists, axis=1)

        # M-step : mettre a jour les centroides
        new_centroids = np.array([
            X[new_labels == k_].mean(axis=0) if (new_labels == k_).sum() > 0
            else centroids[k_]
            for k_ in range(k)
        ])

        if np.allclose(centroids, new_centroids, atol=1e-6):
            print(f"  Converge en {it+1} iterations")
            break
        centroids, labels = new_centroids, new_labels

    inertia = sum(np.linalg.norm(X[labels==k_] - centroids[k_])**2
                  for k_ in range(k) if (labels==k_).sum() > 0)
    return labels, centroids, inertia

X_blobs, y_true = make_blobs(n_samples=400, centers=4, cluster_std=0.9, random_state=42)

print("=== K-Means from scratch ===")
labels, centroids, inertia = kmeans(X_blobs, k=4)
sil = silhouette_score(X_blobs, labels)
print(f"  Inertia   : {inertia:.2f}")
print(f"  Silhouette: {sil:.4f}  (sklearn: {silhouette_score(X_blobs, KMeans(4,n_init=10,random_state=42).fit_predict(X_blobs)):.4f})")


In [ ]:
import numpy as np
from sklearn.cluster import KMeans, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score
from sklearn.datasets import make_blobs, make_moons
import warnings; warnings.filterwarnings("ignore")

np.random.seed(42)

# ============================================================
# ELBOW METHOD + SILHOUETTE -- choisir k
# ============================================================
X_blobs, _ = make_blobs(n_samples=400, centers=4, cluster_std=0.9, random_state=42)

print("=== Elbow method + Silhouette ===")
print(f"  {'k':>4}  {'inertia':>10}  {'silhouette':>12}  {'davies_bouldin':>15}")
from sklearn.metrics import davies_bouldin_score
for k in range(2, 9):
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X_blobs)
    sil = silhouette_score(X_blobs, km.labels_)
    db  = davies_bouldin_score(X_blobs, km.labels_)
    print(f"  {k:>4}  {km.inertia_:>10.1f}  {sil:>12.4f}  {db:>15.4f}")
print("  => Chercher le 'coude' sur inertie + max silhouette + min Davies-Bouldin")

# ============================================================
# DBSCAN -- clusters non spheriques
# ============================================================
print("\n=== DBSCAN vs K-Means sur donnees en lune ===")
X_moons, _ = make_moons(n_samples=300, noise=0.08, random_state=42)

km_labels = KMeans(n_clusters=2, n_init=10, random_state=42).fit_predict(X_moons)
print(f"  K-Means silhouette : {silhouette_score(X_moons, km_labels):.4f}  (mauvais sur formes complexes)")

for eps, min_s in [(0.15, 5), (0.20, 5), (0.25, 5)]:
    db_labels = DBSCAN(eps=eps, min_samples=min_s).fit_predict(X_moons)
    n_cl = len(set(db_labels)) - (1 if -1 in db_labels else 0)
    n_out = (db_labels == -1).sum()
    sil = silhouette_score(X_moons, db_labels) if n_cl > 1 else -99
    print(f"  DBSCAN(eps={eps}, min={min_s}): clusters={n_cl}, outliers={n_out}, sil={sil:.4f}")

# ============================================================
# GMM vs K-Means
# ============================================================
print("\n=== GMM vs K-Means ===")
X_g, _ = make_blobs(n_samples=300, centers=3, cluster_std=[0.5, 1.5, 0.8], random_state=42)

km  = KMeans(n_clusters=3, n_init=10, random_state=42)
gmm = GaussianMixture(n_components=3, covariance_type="full", random_state=42)

km_l  = km.fit_predict(X_g)
gmm_l = gmm.fit_predict(X_g)

print(f"  K-Means  silhouette: {silhouette_score(X_g, km_l):.4f}")
print(f"  GMM      silhouette: {silhouette_score(X_g, gmm_l):.4f}")
print(f"  GMM BIC : {gmm.bic(X_g):.1f}  (pour choisir nb de composantes)")
proba_sample = gmm.predict_proba(X_g[:1])[0]
print(f"  GMM soft assignment (1er point): {proba_sample.round(3)}")
print("  => GMM est probabiliste: chaque point a une proba d appartenance a chaque cluster")


---
## 5. Bagging & Random Forest <a name='bagging'></a>

### Bagging (Bootstrap Aggregating)
- Entraine **B modeles** sur **B sous-echantillons bootstrap** (tirage avec remise)
- Combine par vote majoritaire (classif) ou moyenne (regression)
- Reduit la **variance** sans augmenter le biais
- Environ 37% des donnees ne sont jamais selectionnees (Out-of-Bag = OOB)

### Random Forest = Bagging + Feature Randomness
- A chaque split, tire aleatoirement **sqrt(p)** features candidates (classif)
- Decorrelation des arbres => meilleure reduction de variance que le Bagging pur

### Feature Importance
| Methode | Principe | Biais |
|---|---|---|
| MDI (Mean Decrease Impurity) | Somme des reductions d impurete liees a la feature | Biaise vers haute cardinalite |
| Permutation | Baisse de perf quand on permute la feature | Plus robuste, reference |
| SHAP | Valeurs de Shapley (theorie des jeux) | Meilleur mais lent |


In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.inspection import permutation_importance
import warnings; warnings.filterwarnings("ignore")

np.random.seed(42)
X, y = make_classification(n_samples=1000, n_features=20, n_informative=8,
                            n_redundant=4, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

# ============================================================
# BAGGING from scratch
# ============================================================
class SimpleBagging:
    def __init__(self, n_estimators=30, max_depth=None):
        self.n = n_estimators
        self.max_depth = max_depth
        self.trees = []

    def fit(self, X, y):
        n_samples = len(X)
        for _ in range(self.n):
            idx = np.random.choice(n_samples, n_samples, replace=True)
            tree = DecisionTreeClassifier(max_depth=self.max_depth, random_state=None)
            tree.fit(X[idx], y[idx])
            self.trees.append(tree)
        return self

    def predict_proba(self, X):
        probas = np.mean([t.predict_proba(X) for t in self.trees], axis=0)
        return probas

    def predict(self, X):
        return self.predict_proba(X).argmax(axis=1)

bag = SimpleBagging(n_estimators=50, max_depth=10).fit(X_tr, y_tr)
print("=== Bagging from scratch ===")
print(f"  Accuracy : {accuracy_score(y_te, bag.predict(X_te)):.4f}")
print(f"  AUC      : {roc_auc_score(y_te, bag.predict_proba(X_te)[:,1]):.4f}")

# ============================================================
# RANDOM FOREST -- Impact du nb d arbres et max_features
# ============================================================
print("\n=== Random Forest : impact des hyperparametres ===")
print(f"  {'Modele':35s}  {'CV_AUC':>8}  {'OOB':>6}")
configs = [
    ("Single Tree",            DecisionTreeClassifier(random_state=42), False),
    ("RF n=10",                RandomForestClassifier(n_estimators=10, random_state=42, oob_score=True), True),
    ("RF n=100",               RandomForestClassifier(n_estimators=100, random_state=42, oob_score=True), True),
    ("RF n=100 max_f=2",       RandomForestClassifier(n_estimators=100, max_features=2, random_state=42, oob_score=True), True),
    ("RF n=100 max_f=sqrt",    RandomForestClassifier(n_estimators=100, max_features="sqrt", random_state=42, oob_score=True), True),
    ("RF n=100 max_f=all",     RandomForestClassifier(n_estimators=100, max_features=None, random_state=42, oob_score=True), True),
]
for name, model, has_oob in configs:
    cv = cross_val_score(model, X_tr, y_tr, cv=5, scoring="roc_auc").mean()
    model.fit(X_tr, y_tr)
    oob = f"{model.oob_score_:.4f}" if has_oob else "  N/A"
    print(f"  {name:35s}  {cv:.4f}  {oob}")


In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance
import warnings; warnings.filterwarnings("ignore")

np.random.seed(42)
X, y = make_classification(n_samples=1000, n_features=15, n_informative=5,
                            n_redundant=2, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

rf = RandomForestClassifier(n_estimators=200, random_state=42, oob_score=True).fit(X_tr, y_tr)
print(f"=== Feature Importance ===  (OOB Score = {rf.oob_score_:.4f})")

# MDI
mdi = rf.feature_importances_
print("\nMDI (top 7) :")
for i in np.argsort(mdi)[::-1][:7]:
    bar = "#" * int(mdi[i]*100)
    print(f"  feature_{i:2d}: {mdi[i]:.4f}  {bar}")

# Permutation (plus robuste)
perm = permutation_importance(rf, X_te, y_te, n_repeats=15, random_state=42)
print("\nPermutation Importance (top 7) :")
for i in np.argsort(perm.importances_mean)[::-1][:7]:
    bar = "#" * int(perm.importances_mean[i]*100)
    print(f"  feature_{i:2d}: {perm.importances_mean[i]:.4f} +/- {perm.importances_std[i]:.4f}  {bar}")

print("\nConseils :")
print("  - MDI rapide mais biaise vers les features a haute cardinalite")
print("  - Permutation: plus lent mais plus robuste, utiliser sur le test set")
print("  - SHAP: gold standard, explique chaque prediction individuellement")


---
## 6. Boosting <a name='boosting'></a>

### Principe general
Entrainer des modeles **sequentiellement** : chaque modele corrige les erreurs du precedent.
Reduit principalement le **biais** (contrairement au Bagging qui reduit la variance).

### AdaBoost
- Pondere les **echantillons mal classes** a chaque iteration
- Prediction finale = vote pondere des weak learners
- Sensible aux **outliers** (ils accumulent du poids)

### Gradient Boosting (GBM)
- Fit le **gradient negatif de la loss** (pseudo-residus) a chaque etape
- `F_m(x) = F_{m-1}(x) + learning_rate * h_m(x)`
- Learning rate faible + beaucoup d arbres = meilleure generalisation

### XGBoost / LightGBM / CatBoost
| Feature | XGBoost | LightGBM | CatBoost |
|---|---|---|---|
| Split | Level-wise | Leaf-wise | Leaf-wise |
| Variables cat. | Encodage manuel | Encodage manuel | Natif |
| Vitesse | Rapide | Tres rapide | Rapide |
| Regularisation | L1+L2 | L1+L2 | Ordinal boosting |


In [ ]:
import numpy as np
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, roc_auc_score
import warnings; warnings.filterwarnings("ignore")

np.random.seed(42)
X, y = make_classification(n_samples=1000, n_features=20, n_informative=10,
                            n_redundant=3, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

# ============================================================
# ADABOOST from scratch
# ============================================================
class SimpleAdaBoost:
    def __init__(self, n_estimators=50):
        self.M = n_estimators
        self.alphas = []
        self.stumps = []

    def fit(self, X, y):
        n = len(X)
        w = np.ones(n) / n

        for _ in range(self.M):
            stump = DecisionTreeClassifier(max_depth=1)
            stump.fit(X, y, sample_weight=w)
            y_pred = stump.predict(X)

            # Erreur ponderee
            err = w[y_pred != y].sum()
            err = np.clip(err, 1e-10, 1 - 1e-10)

            # Poids du stump
            alpha = 0.5 * np.log((1 - err) / err)

            # Mise a jour des poids (y in {-1, +1} pour AdaBoost)
            y_signed = 2 * y - 1
            yp_signed = 2 * y_pred - 1
            w *= np.exp(-alpha * y_signed * yp_signed)
            w /= w.sum()

            self.alphas.append(alpha)
            self.stumps.append(stump)
        return self

    def predict(self, X):
        scores = sum(a * (2*s.predict(X)-1) for a, s in zip(self.alphas, self.stumps))
        return (scores > 0).astype(int)

ada_scratch = SimpleAdaBoost(n_estimators=100).fit(X_tr, y_tr)
print("=== AdaBoost from scratch ===")
print(f"  Accuracy : {accuracy_score(y_te, ada_scratch.predict(X_te)):.4f}")

# ============================================================
# COMPARAISON BOOSTING
# ============================================================
print("\n=== Comparaison Boosting ===")
from sklearn.ensemble import RandomForestClassifier
models = [
    ("Single Tree (depth=3)",       DecisionTreeClassifier(max_depth=3, random_state=42)),
    ("AdaBoost (n=100)",            AdaBoostClassifier(n_estimators=100, random_state=42, algorithm="SAMME")),
    ("GradientBoosting (n=100)",    GradientBoostingClassifier(n_estimators=100, lr=0.1, max_depth=3, random_state=42)),
    ("RandomForest (n=100)",        RandomForestClassifier(n_estimators=100, random_state=42)),
]
print(f"  {'Modele':35s}  {'CV_AUC':>8}  {'test_AUC':>9}")
for name, model in models:
    cv = cross_val_score(model, X_tr, y_tr, cv=5, scoring="roc_auc").mean()
    model.fit(X_tr, y_tr)
    if hasattr(model, "predict_proba"):
        te_auc = roc_auc_score(y_te, model.predict_proba(X_te)[:,1])
    else:
        te_auc = roc_auc_score(y_te, model.decision_function(X_te))
    print(f"  {name:35s}  {cv:.4f}  {te_auc:.4f}")


In [ ]:
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, validation_curve
from sklearn.metrics import roc_auc_score
import warnings; warnings.filterwarnings("ignore")

np.random.seed(42)
X, y = make_classification(n_samples=1000, n_features=20, n_informative=10, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

# ============================================================
# GRADIENT BOOSTING -- tuning hyperparametres
# ============================================================
print("=== GBM : learning_rate vs n_estimators (tradeoff) ===")
print("  Regle : learning_rate faible + n_estimators eleve => meilleure generalisation")
print()
configs = [
    (0.5, 50),
    (0.1, 100),
    (0.1, 200),
    (0.05, 300),
    (0.01, 500),
]
print(f"  {'lr':>6}  {'n_est':>6}  {'train_AUC':>10}  {'test_AUC':>10}")
for lr, n_est in configs:
    gbm = GradientBoostingClassifier(n_estimators=n_est, learning_rate=lr,
                                      max_depth=3, subsample=0.8, random_state=42)
    gbm.fit(X_tr, y_tr)
    tr_auc = roc_auc_score(y_tr, gbm.predict_proba(X_tr)[:,1])
    te_auc = roc_auc_score(y_te, gbm.predict_proba(X_te)[:,1])
    print(f"  {lr:>6.2f}  {n_est:>6d}  {tr_auc:>10.4f}  {te_auc:>10.4f}")

print("\n=== Early Stopping (eviter overfitting) ===")
gbm_es = GradientBoostingClassifier(n_estimators=500, learning_rate=0.05,
                                     max_depth=3, validation_fraction=0.1,
                                     n_iter_no_change=20, random_state=42)
gbm_es.fit(X_tr, y_tr)
print(f"  Arbres utilises (early stop) : {gbm_es.n_estimators_}")
print(f"  AUC : {roc_auc_score(y_te, gbm_es.predict_proba(X_te)[:,1]):.4f}")

print("\n=== Hyperparametres importants GBM ===")
print("  n_estimators      : nb d arbres. Augmenter si lr faible")
print("  learning_rate     : shrinkage. Typiquement 0.01-0.1")
print("  max_depth         : profondeur. 3-5 en general")
print("  subsample         : fraction des donnees par arbre (< 1 = stochastic GBM)")
print("  min_samples_leaf  : regularisation sur les feuilles")
print("  max_features      : fraction des features par split")


---
## 7. Métriques & Évaluation <a name='metrics'></a>

### Cross-Validation
- **k-Fold** : divise en k folds, entraine sur k-1, test sur 1 => k repetitions
- **Stratified k-Fold** : preserve la distribution des classes (important si desequilibre)
- **LOOCV** : Leave-One-Out, extremement lent mais sans biais
- **TimeSeriesSplit** : pour donnees temporelles (pas de fuite du futur)

### Grid Search vs Random Search vs Bayesian Optimization
| Methode | Avantage | Inconvenient |
|---|---|---|
| GridSearch | Exhaustif | Exponentiel |
| RandomSearch | Efficace sur les distributions continues | Pas optimal |
| BayesianOpt | Informe par les iterations precedentes | Plus complexe |


In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import (KFold, StratifiedKFold, cross_val_score,
                                      GridSearchCV, RandomizedSearchCV)
from sklearn.datasets import make_classification
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                               f1_score, roc_auc_score, average_precision_score)
from scipy.stats import randint, uniform
import warnings; warnings.filterwarnings("ignore")

np.random.seed(42)
X, y = make_classification(n_samples=800, n_features=15, n_informative=6,
                            weights=[0.8, 0.2], random_state=42)

# ============================================================
# CROSS-VALIDATION STRATEGIES
# ============================================================
print("=== Strategies de Cross-Validation ===")
rf = RandomForestClassifier(n_estimators=50, random_state=42)

kf     = KFold(n_splits=5, shuffle=True, random_state=42)
skf    = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, cv in [("KFold", kf), ("StratifiedKFold", skf)]:
    scores = cross_val_score(rf, X, y, cv=cv, scoring="roc_auc")
    print(f"  {name:20s}: {scores.mean():.4f} +/- {scores.std():.4f}")
    print(f"  {'':20s}  fold scores: {scores.round(4)}")

# ============================================================
# GRID SEARCH vs RANDOM SEARCH
# ============================================================
print("\n=== Grid Search ===")
param_grid = {
    "n_estimators": [50, 100],
    "max_depth": [3, 5, None],
    "min_samples_leaf": [1, 5]
}
gs = GridSearchCV(RandomForestClassifier(random_state=42),
                  param_grid, cv=5, scoring="roc_auc", n_jobs=-1)
gs.fit(X, y)
print(f"  Best params : {gs.best_params_}")
print(f"  Best AUC    : {gs.best_score_:.4f}")
print(f"  Nb de fits  : {len(gs.cv_results_['mean_test_score'])} * 5 folds")

print("\n=== Random Search ===")
param_dist = {
    "n_estimators": randint(50, 300),
    "max_depth": [3, 5, 8, None],
    "min_samples_leaf": randint(1, 20),
    "max_features": uniform(0.1, 0.9),
}
rs = RandomizedSearchCV(RandomForestClassifier(random_state=42),
                        param_dist, n_iter=20, cv=5,
                        scoring="roc_auc", random_state=42, n_jobs=-1)
rs.fit(X, y)
print(f"  Best params : {rs.best_params_}")
print(f"  Best AUC    : {rs.best_score_:.4f}")
print(f"  Nb de fits  : 20 * 5 folds (vs {2*3*2}*5 pour Grid)")


---
## 8. Questions d’entretien <a name='questions'></a>

**Q: Difference entre Bagging et Boosting ?**
> Bagging = modeles independants en parallele, reduit la **variance**. Boosting = modeles sequentiels, chacun corrige le precedent, reduit le **biais**. RF est du Bagging. XGBoost est du Boosting.

**Q: Pourquoi Random Forest est moins sujet a l'overfitting qu'un arbre unique ?**
> La combinaison de B arbres decorrelees (grace au feature sampling) lisse les predictions. La variance de la moyenne de B variables independantes est sigma^2/B.

**Q: Quand choisir Lasso plutot que Ridge ?**
> Lasso si on suppose que peu de features sont vraiment importantes (sparse solution). Ridge si toutes les features contribuent. ElasticNet si features correlees entre elles.

**Q: Expliquer le Gradient Boosting en une phrase.**
> On ajoute iterativement des arbres qui fittent le **gradient negatif de la loss** (les pseudo-residus), ce qui permet de descendre dans l'espace des fonctions plutot que dans l'espace des parametres.

**Q: Comment detecter et gerer le class imbalance ?**
> Detection : regarder distribution des classes. Gestion : (1) Resampling (SMOTE, undersampling), (2) `class_weight='balanced'`, (3) ajuster le seuil de decision, (4) utiliser PR-AUC plutot que AUC-ROC.

**Q: Quelle est la difference entre precision et recall, et quand favoriser l'un ou l'autre ?**
> Precision = parmi les positifs predits, combien sont vrais (eviter faux positifs). Recall = parmi les vrais positifs, combien detectes (eviter faux negatifs). Fraude/cancer => recall. Spam/recommandation => precision.

**Q: DBSCAN vs K-Means : quand utiliser lequel ?**
> K-Means si clusters spheriques de taille similaire, k connu a l'avance, dataset large. DBSCAN si clusters de forme arbitraire, detection d'outliers necessaire, k inconnu.

**Q: Comment gerer des donnees temporelles dans la validation ?**
> Utiliser `TimeSeriesSplit` : toujours entrainer sur le passe, tester sur le futur. Ne jamais melanger les folds. Attention a la data leakage avec les features fenetre glissante.
